# 运行时的上下文

比较特别，不像长期记忆和短期记忆，他们是持久化的

运行时上下文，只使用一次，就是运行图的时候，可以额外传递上下文的参数，这个上下文结构可以自由定义，只要是dataclass数据的类，都可以，里面可以有属性，可以有值，图就可以在任意一个节点获取到环境上下文的信息，

1. 只对本次调用生效，也不会被持久化，
2. 也不会在同一个thread_id的下一次中自动恢复， 下次调用也不能调用到上次的内容


给一个案例，不同的用户调用同一个机器人智能体，机器人运行时根据context中的用户名和会员等级生成不同风格的回复，见人说人话，看见vip等级高更谄媚一点

In [ ]:
from calendar import c
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek

from loguru import logger
from dotenv import load_dotenv
from torch import mode
load_dotenv(override=True)
from typing import Literal
from langgraph.runtime import Runtime
from langgraph.graph.message import MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.postgres import PostgresSaver
from dataclasses import dataclass


model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义运行时的环境上下文
@dataclass
class UserContext: 
    username: str
    membership_level: str

# 2. 定义状态
class OverAllState(MessagesState):
    user_input: str
    output: str

# 3. 定义节点
def llm_node(state: OverAllState, runtime: Runtime[UserContext]) -> OverAllState:
    # 1. 获取环境上下文，判断当前的用户等级
    runtime_context = runtime.context
    level = runtime_context.membership_level

    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username

        logger.info(f"当前用户:{username},会员等级:{level}")

        if level == "VIP":
            system_prompt = f"你是高级客户助理，当前VIP用户是{username},请使用尊称'您'，语气热情周到，回复末尾加上'VIP🥇服务'"
        else:
            system_prompt = f"你是普通客户助理,当前用户是{username}, 请友好简洁回复问题"

    else:
        system_prompt = f"你是普通客户助理，请友好简洁回复问题"

    user_input = state["user_input"]
    messages = state.get("messages", [])
    response = model.invoke([SystemMessage(content=system_prompt)] + messages + [HumanMessage(content=user_input)]).content

    return {
        "messages": messages,
        "output": response
    }

            
# 4. 构建图
builder = StateGraph(state_schema=OverAllState, context_schema=UserContext)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

# ====================第一次调用：传入VIP上下文用户========================
res = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice", membership_level="VIP")
    )

print(res)

# ====================第二次调用：传入普通上下文用户========================
res1 = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice", membership_level="普通用户")
    )

print(res1)

2026-09-16 17:52:42.267 | INFO     | __main__:llm_node:50 - 当前用户:Alice,会员等级:VIP


2026-09-16 17:52:43.664 | INFO     | __main__:llm_node:50 - 当前用户:Alice,会员等级:普通用户


{'messages': [], 'user_input': '你好，帮我查一下最近有什么优惠活动', 'output': '您好，Alice！非常感谢您的咨询！😊 我马上为您查询最近的最新优惠活动，请您稍等片刻哦～\n\n正在为您查询中...🔍\n\n经过查询，目前我们为VIP客户准备了以下优惠活动：\n\n🎉 **VIP专属优惠活动**：\n1. **限时折扣**：部分精选商品享受8折优惠，活动截止至本月底\n2. **满减活动**：单笔消费满500元立减100元\n3. **积分翻倍**：VIP客户本月消费积分双倍累积\n4. **生日礼遇**：如您本月生日，可享受额外专属礼品\n\n具体活动详情和适用商品，我可以为您进一步详细介绍。请问您对哪项活动比较感兴趣呢？或者您想了解某个特定品类的优惠信息吗？我很乐意为您提供更精准的服务！💝\n\nVIP🥇服务'}
{'messages': [], 'user_input': '你好，帮我查一下最近有什么优惠挥动', 'output': '你好，Alice！目前我们有一些优惠活动，不过具体内容可能会根据时间和地区有所不同。你可以告诉我你感兴趣的产品或服务，我帮你查一下相关的优惠信息哦！'}


UserContext 上下文，直接在llm_node定义节点的时候，定义Runtime[xxxx]就可以使用了

然后运行的时候要传入

哪些需要传入
适合放入运行时上下文：
1. 当前登录用户信息
2. 请求来源（Web / API / 小程序）
3. 调用方标识、Trace ID 
4. 本次调用的功能开关 

不适合（应放入图状态）
1. 需要在多轮对话间共享的数据
2. 需要在检查点中恢复的执行进度
3. 需要跨调用持久化的业务数据
4. 节点间需要传递的计算结果


总结：需要共享、持久化的放State；仅当前调用有效的放Runtime Context


The user asks: what does "data shared across multi-turn conversations" mean? Is one turn of conversation started by one thread_id?

Need to explain thread_id = session/conversation, not a single turn. One thread_id covers many turns (many invoke calls). State (messages) persists across invokes with same thread_id via checkpointer. Context is per-invoke.
好问题！你笔记里那句“**需要在多轮对话间共享的数据**（应放入图状态，而不是 context）”，关键在于理清三个概念：**一轮对话、一次 invoke、一个 thread_id**。结合你笔记里的实际代码来解释。

## 1. 核心结论：thread_id 对应的是“一整段会话”，不是“一轮对话”

层级关系是这样的：

```
一个 thread_id = 一段完整会话（session）
 └── 包含多轮对话（turn）
      └── 每一轮 = 一次 graph.invoke() 调用
           └── 内部执行 N 个节点/超步
```

- **一轮对话** = 用户问一次 + 模型答一次 = **一次** `graph.invoke()`
- **一个 thread_id** = 把**很多次** `invoke` 串起来的会话标识

你在 `01_in_memory.ipynb` 里早就用过这个机制：

```python
config = {"configurable": {"thread_id": "chapter03-01"}}

# 第 1 轮对话（第 1 次 invoke）
graph.invoke({"messages": [HumanMessage("你好，我是老王")]}, config=config)

# 第 2 轮对话（第 2 次 invoke，同一个 thread_id！）
graph.invoke({"messages": [HumanMessage("我叫什么名字？")]}, config=config)
# 模型能答出"你是老王" —— 因为两次调用通过同一个 thread_id 共享了 state
```

**同一个 thread_id 的两次 invoke 之间，消息（state）被 checkpointer 持久化并自动加载**——这就是“多轮对话间共享的数据”。

## 2. 为什么笔记说这类数据该放 state，不该放 context

对照你 `11_context.ipynb` 里的代码，注意一个细节：这个图**根本没有配 checkpointer，也没传 config**：

```python
graph = builder.compile()   # 没有 checkpointer

res = graph.invoke(
    {"user_input": "你好，帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice", membership_level="VIP")  # context 是 invoke 的参数
)

res1 = graph.invoke(
    {"user_input": "..."},
    context=UserContext(username="Alice", membership_level="普通用户")  # 下一次要重新传
)
```

两次 invoke 的行为对比：

| 维度 | state（图状态） | context（运行时上下文） |
|---|---|---|
| 传入方式 | 每次只传增量输入，历史由 checkpointer 自动恢复 | 每次 invoke 都要**完整重新传** |
| 两次 invoke 之间 | ✅ 同一 thread_id 自动延续（配了 checkpointer 时） | ❌ 上次的 context 直接消失，不持久化 |
| 存活范围 | 跨越整个会话（多轮） | 仅本次 invoke |
| 典型内容 | messages、计算结果、执行进度 | 用户身份、trace_id、功能开关 |

假设你想让这个客服机器人**记住 Alice 上一轮问了什么**（VIP 用户第二轮说“再详细说说第 2 条”）：

- 如果靠 context 传递 → 每轮都要把全部历史手动拼进 context，很容易漏
- 正确做法：配 checkpointer + 同一个 thread_id，`messages` 存在 state 里自动跨轮延续

所以笔记那条“**不适合放入 context 的：需要在多轮对话间共享的数据**”，意思就是：**凡是希望在“同一个 thread_id 的下一次 invoke”里还能用到的数据，都放 state；只有“本次调用专属、用完即弃”的信息（登录身份、trace id、本次开关）才放 context**。

## 3. 一个容易混淆的点：context 也可以“每一轮都传同一个值”

注意区分“每轮都传”和“跨轮共享”：

```python
# 每一轮都手动传 context —— 这不是"共享"，只是"每次都重新提供"
graph.invoke(input1, context=UserContext("Alice", "VIP"), config=config)
graph.invoke(input2, context=UserContext("Alice", "VIP"), config=config)
```

这种写法下，VIP 身份确实“每轮都在”，但那是**调用方每轮自己负责传的**；一旦某一轮忘了传或传了不同的值，没有任何机制帮你记着。而 state 是 checkpointer **自动**替你保留的。生产代码里“用户信息”常常每轮都要传（因为登录态由 HTTP 请求带来），这没问题——但**对话历史、中间计算结果**绝不能靠 context 手传，必须走 state。

## 4. 结合你学的三种记忆，一张总表

| 机制 | 生命周期 | 标识/载体 | 例子 |
|---|---|---|---|
| 短期记忆（state + checkpointer） | 同一 thread_id 内跨多轮 invoke | `thread_id` | 对话历史 messages |
| 长期记忆（Store） | 跨 thread_id、跨进程、永久 | `namespace + key` | 用户偏好 |
| 运行时上下文（context） | 仅一次 invoke | 无（每次调用传入） | username、VIP 等级、trace_id |

> 💡 **一句话回答你的问题**：一轮对话 = 一次 `invoke`；一个 thread_id = 一段包含多轮对话的会话。“多轮对话间共享的数据”就是指在同一 thread_id 下、多次 invoke 之间需要延续的信息（最典型就是 messages），这类数据 LangGraph 通过 checkpointer 存在 **state** 里自动恢复，而 context 每次调用就蒸发，所以不能承担这个职责。

## 4. 结合你学的三种记忆，一张总表

| 机制 | 生命周期 | 标识/载体 | 例子 |
|---|---|---|---|
| 短期记忆（state + checkpointer） | 同一 thread_id 内跨多轮 invoke | `thread_id` | 对话历史 messages |
| 长期记忆（Store） | 跨 thread_id、跨进程、永久 | `namespace + key` | 用户偏好 |
| 运行时上下文（context） | 仅一次 invoke | 无（每次调用传入） | username、VIP 等级、trace_id |

不乱，这两个其实是**同一套机制的两种触发场景**。先给核心结论，再分情况总结。

## 核心结论

**是的**——只要编译图时配了 `checkpointer`，并且每次调用带**同一个 `thread_id`**，那么这个线程的状态（包括全部历史 messages）就会在每次超步结束后被自动持久化，下次调用时自动恢复。你根本不需要“手动维持”历史，`checkpointer + thread_id` 就是干这个的。

**而“传 None 恢复”并不是另一种持久化机制，它只是同一个机制在“上次执行中断”时的特殊用法。**

## 情况一：多轮对话，每次传入新问题（最常用）

这是 `01_in_memory.ipynb` 里的标准用法：

```python
config = {"configurable": {"thread_id": "chapter03-01"}}

# 第 1 轮：正常传新输入
graph.invoke({"messages": [HumanMessage("你好，我是老王")]}, config=config)

# 第 2 轮：继续传新输入，同一个 thread_id
graph.invoke({"messages": [HumanMessage("我叫什么名字？")]}, config=config)
# 模型答"老王" —— 上一轮的 messages 被自动加载拼接了

# 第 3、4、5... 轮同理，想聊多少轮都行
```

工作原理：

```
invoke(输入1) → 执行图 → 每个超步写检查点 → 返回结果（state 里累积了输入1+回复1）
invoke(输入2) → 从检查点恢复 state → 追加输入2 → 模型看到完整历史 → 回复2
invoke(输入3) → 同上，历史越来越长 ...
```

要点：
- 每次传的都是**新输入**（增量），不是全量历史
- 历史的累积、拼接由 `MessagesState` 的 `add_messages` reducer + checkpointer 自动完成
- 换一个 `thread_id`，就是一段全新的会话（比如换了一个用户）

## 情况二：失败/中断后恢复，传入 None

这是 `05_error.ipynb` 里的用法。前提是上次执行**中途抛异常**了（比如 `node_joke` 人为抛错）：

```python
# 第一次执行：node_poem 成功，node_joke 抛异常，图中断
try:
    graph.invoke({"topic": "猫"}, config=config)
except Exception:
    pass

# 恢复执行：输入传 None + 同一个 thread_id + 不带 checkpoint_id
graph.invoke(None, config=config)
```

`None` 的含义是：**“我不提供新输入，请你从上次中断的检查点继续”**。LangGraph 会：

1. 找到该 thread 最新的检查点
2. 发现有些节点没执行完（`node_joke` 失败了）
3. **只重跑失败的任务**：`node_poem` 的写入结果已提交到检查点，被当作“缓存写入”直接复用，不重跑
4. 继续往后执行 `node_output` 直到 END

## 两种情况对比总结

| 维度 | 情况一：多轮对话 | 情况二：失败恢复 |
|---|---|---|
| 上次执行状态 | 正常跑到了 END | 中途异常中断 |
| 输入 | 新的问题/数据 | `None`（表示不提供新输入） |
| LangGraph 的行为 | 从最新检查点恢复 state，**追加新输入，重新从头开始走图** | 从最新检查点恢复，**不追加输入，续跑未完成的节点** |
| 成功过的节点 | 重新执行（因为是新一轮运行） | 不重跑（复用检查点里的缓存写入） |
| 使用场景 | 聊天机器人、多轮问答 | 故障恢复、人工审批后继续 |

## 一个统一的心智模型

把 checkpointer 想成“游戏存档”，`thread_id` 是存档槽位，每次 `invoke` 开始时 LangGraph 都会先读存档：

- **存档显示“已通关”（正常结束）** → 你传了新输入，就开启新一关继续玩（情况一）
- **存档显示“打 Boss 打一半死了”（中断）** → 你传 `None`，就从死掉的地方原地复活接着打（情况二）
- 你**不能**给“已通关”的存档传 `None`——没有未完成的任务可恢复，等于无事发生/报错；你也**不应该**给“打一半”的存档传新输入去开新局——部分 LangGraph 版本会报错或行为未定义

> 💡 **一句话总结**：`checkpointer + 同一 thread_id` 始终在“维持历史”——这一点两种情况完全一样。唯一的区别在于**图上次是否跑完**：跑完了，传新输入就是多轮对话；没跑完，传 `None` 就是断点续跑。同一个机制，两种触发姿势而已。